# YOLO-World · VisDrone 5 × 5 training/inference overlap grid

Run this notebook **after** `00_data_and_protocol_SageMaker_VisDrone.ipynb`, in the same persistent SageMaker JupyterLab working directory. It uses the finalized ClearML Dataset ID and the identical frozen protocol, prompts, base/novel categories and image selections from the input notebook.

The requested experiment trains **five fresh YOLO-World models**, using training tile overlap **10%, 20%, 30%, 40%, 50%** in both axes. Every model is tested with sliced inference at all five overlaps: **25 fine-tuned sliced conditions**. Each model starts from the same unmodified pretrained checkpoint, not the preceding fine-tuned model. All original model/training/inference settings remain unchanged: 640 × 640 tiles, input size 1280, 80 epochs in full mode, batch 8, AdamW, learning rate 0.0002, seed 42, prompts, confidence floors, NMS and IoU thresholds. Whole base-safe training images and positive tiles are both retained; model-selection validation remains whole-image and base-safe.

The original four-condition zero-shot/whole/sliced sweep is replaced by this requested 25-condition grid, so it does not add extra tests. The original whole-frame inference helpers and pretrained checkpoint remain available. The configured evaluation split is still **val**; switch `eval_split` to `test` only after freezing choices, as in the original notebook. Every condition uses the exact same evaluation image list.

With `report_ground_truth_objects=True`, results include GT objects, predicted objects, correctly detected objects (one-to-one, same class, confidence ≥0.25 and IoU ≥0.5), missed objects and false positives, overall, per class and per image. Ignore IDs 0 and 11 remain excluded under the original convention. This adds object-count reporting without replacing AP/AR or changing matching thresholds.

With `save_comparison_images=True`, **every evaluated image in every condition** is saved as a high-quality JPEG: ground-truth boxes on the left; model boxes, class names and confidence on the right. Visualization uses the existing operating threshold (0.25); AP still uses the original extraction floor (0.001). Images are saved after inference, outside the timed pipeline, under `results/yolo_world_v2/<run_id>/comparisons/<condition>/`.

For a quick check use the original `run_mode="smoke"`; these numbers are not final paper results. Full mode uses the complete configured split. Large weights, predictions and comparison images stay local; ClearML receives compact tables and inventory only. Each overlap gets its own training view and checkpoint filenames, so results cannot overwrite another overlap.


In [1]:
# Restart the kernel after the first installation if prompted, then run all cells top-to-bottom.
%pip install -q "ultralytics>=8.3,<9" "clearml>=1.16,<3" "pycocotools>=2.0.7,<3" "pillow>=10,<13" "numpy>=1.24,<3" "pandas>=2,<3" "tqdm>=4.66,<5"


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass

os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"

os.environ["CLEARML_API_ACCESS_KEY"] = getpass("ClearML access key: ")
os.environ["CLEARML_API_SECRET_KEY"] = getpass("ClearML secret key: ")

print("ClearML credentials set for this kernel.")

ClearML access key:  ········
ClearML secret key:  ········


ClearML credentials set for this kernel.


In [3]:
import csv
import gc
import hashlib
import json
import os
import platform
import random
import shutil
import time
import uuid
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from tqdm.auto import tqdm

@dataclass
class Config:
    # The project root is the SAME current directory used by notebook 00.
    project_root: str = "SageMaker_VisDrone_OVD_2026"
    project_name: str = "UAV-OVD-Research"
    task_name_prefix: str = "Experiment 1 - Aerial OVD Characterisation"
    clearml_project: str = "SageMaker-Aerial-OVD-2026/YOLO-World-VisDrone-FT-Whole-Sliced-2026"
    yolo_checkpoint: str = "yolov8s-worldv2.pt"
    grounding_dino_checkpoint: str = "IDEA-Research/grounding-dino-tiny"

    run_mode: str = "full"                  # "smoke" or "full".
    eval_split: str = "val"                  # Switch to "test" only after freezing ALL choices.
    smoke_images: int = 5
    epochs: int = 80
    smoke_epochs: int = 1
    batch: int = 8
    yolo_image_size: int = 1280
    yolo_extraction_threshold: float = 0.001
    yolo_local_nms_iou: float = 0.70
    operating_threshold_yolo: float = 0.25
    operating_threshold_gdino: float = 0.25
    slice_height: int = 256
    slice_width: int = 256
    overlap_height_ratio: float = 0.20
    overlap_width_ratio: float = 0.20
    # Both axes use each grid value. Legacy single-overlap defaults above stay unchanged.
    train_overlap_ratios: tuple = (0.10, 0.20, 0.30, 0.40, 0.50)
    eval_overlap_ratios: tuple = (0.10, 0.20, 0.30, 0.40, 0.50)
    report_ground_truth_objects: bool = True
    save_comparison_images: bool = True
    include_whole_prediction_in_sliced: bool = False
    global_nms_iou: float = 0.50
    max_detections_per_image: int = 500
    evaluation_iou: float = 0.50
    seed: int = 42
    save_to_google_drive: bool = False
    drive_output_folder: str = "MyDrive/UAV_OVD_Research/Experiment_1"
    skip_completed_conditions: bool = True
    register_manifest_as_clearml_dataset: bool = True
    upload_full_raw_dataset_to_clearml: bool = False
    verify_image_hashes: bool = True

CFG = Config()
assert tuple(CFG.train_overlap_ratios) == (0.10, 0.20, 0.30, 0.40, 0.50)
assert tuple(CFG.eval_overlap_ratios) == (0.10, 0.20, 0.30, 0.40, 0.50)

def overlap_tag(value):
    return f"overlap_{int(round(value * 100)):02d}"

assert CFG.run_mode in {"smoke", "full"}
assert CFG.eval_split in {"val", "test"}
assert 0 <= CFG.yolo_extraction_threshold <= CFG.operating_threshold_yolo < 1
assert 0 <= CFG.overlap_height_ratio < 1 and 0 <= CFG.overlap_width_ratio < 1
assert CFG.slice_height >= 64 and CFG.slice_width >= 64
assert 0 <= CFG.yolo_local_nms_iou <= 1
assert 0 <= CFG.global_nms_iou <= 1
assert 0 <= CFG.evaluation_iou <= 1
if CFG.eval_split == "test" and CFG.run_mode != "full":
    raise ValueError("Reserved test-dev evaluation must use the full protocol, never a smoke subset.")
if not torch.cuda.is_available():
    raise RuntimeError("Use a SageMaker GPU kernel for this train-and-evaluate notebook.")

random.seed(CFG.seed)
np.random.seed(CFG.seed)
torch.manual_seed(CFG.seed)
torch.cuda.manual_seed_all(CFG.seed)
ROOT = (Path.cwd().resolve() / CFG.project_root)
DATA_ROOT = ROOT / "visdrone_det_original"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
RUN_DIR = ROOT / "runs" / "yolo_world_v2" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
WEIGHTS_DIR = ROOT / "weights" / "yolo_world_v2" / RUN_ID
RESULTS_DIR = ROOT / "results" / "yolo_world_v2" / RUN_ID
WEIGHTS_DIR.mkdir(parents=True, exist_ok=False)
RESULTS_DIR.mkdir(parents=True, exist_ok=False)
YOLO_CONFIG_DIR = RUN_DIR / "ultralytics_config"
YOLO_CONFIG_DIR.mkdir()
os.environ["YOLO_CONFIG_DIR"] = str(YOLO_CONFIG_DIR)
print("Run:", RUN_ID, "| Device:", torch.cuda.get_device_name(0))
print("Local training workspace:", RUN_DIR)
print("Persistent local weights:", WEIGHTS_DIR)
print("Persistent local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("New ClearML project:", CFG.clearml_project)


Run: 20261008T163054Z_73692224 | Device: Tesla T4
Local training workspace: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yolo_world_v2/20261008T163054Z_73692224
Persistent local weights: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/weights/yolo_world_v2/20261008T163054Z_73692224
Persistent local results: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/results/yolo_world_v2/20261008T163054Z_73692224
Free space (GiB): 19.98
New ClearML project: SageMaker-Aerial-OVD-2026/YOLO-World-VisDrone-FT-Whole-Sliced-2026


## 1 · Load the *published* protocol, not a new random split

Do not modify the v1 Dataset. The validation images with annotated novel targets are omitted **only during training model selection**; the evaluation below still includes all original ground-truth categories. The local SHA-256 checks make accidental substitutions or edits visible. Never train on val/test-dev.


In [4]:
from clearml import Dataset, Task

id_file = ROOT / "clearml_protocol_dataset_id.txt"
if not id_file.is_file():
    raise FileNotFoundError(f"Run notebook 00 first; missing {id_file}")
PROTOCOL_ID = id_file.read_text(encoding="utf-8").strip()
ds = Dataset.get(dataset_id=PROTOCOL_ID)
if not ds.is_final():
    raise RuntimeError("Protocol Dataset is not finalized. Do not train on mutable splits.")
PROTOCOL_DIR = Path(ds.get_local_copy()).resolve()

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

required = {"categories_and_prompts.json", "protocol.json", "summary.json",
            "train.csv", "val.csv", "test.csv", "train_base_safe.csv", "protocol_fingerprint.txt"}
if not required.issubset({p.name for p in PROTOCOL_DIR.iterdir() if p.is_file()}):
    raise RuntimeError("ClearML protocol Dataset is missing a required v1 file.")
digest = hashlib.sha256()
for name in sorted(required - {"protocol_fingerprint.txt"}):
    content = (PROTOCOL_DIR / name).read_bytes()
    digest.update(name.encode() + b"\x00" + content + b"\x00")
FINGERPRINT = digest.hexdigest()
if FINGERPRINT != (PROTOCOL_DIR / "protocol_fingerprint.txt").read_text().strip():
    raise RuntimeError("Protocol contents do not match their published fingerprint.")
meta_fingerprint = (ds.get_metadata() or {}).get("protocol_fingerprint")
if meta_fingerprint != FINGERPRINT:
    raise RuntimeError("ClearML Dataset metadata fingerprint does not match its files.")
spec = json.loads((PROTOCOL_DIR / "protocol.json").read_text(encoding="utf-8"))
cats = json.loads((PROTOCOL_DIR / "categories_and_prompts.json").read_text(encoding="utf-8"))
if spec["version"] != "1.0.0":
    raise RuntimeError("This notebook expects the published protocol v1.0.0.")
CATEGORY_IDS = sorted(int(k) for k in cats["official_category_id_to_name"])
CATEGORY_NAMES = [cats["official_category_id_to_name"][str(k)] for k in CATEGORY_IDS]
PROMPTS = [cats["prompt_by_category_name"][name] for name in CATEGORY_NAMES]
BASE_NAMES = cats["base_category_names"]
NOVEL_NAMES = cats["novel_category_names"]
BASE_IDS = set(cats["base_official_ids"])
NOVEL_IDS = set(cats["novel_official_ids"])
assert len(CATEGORY_IDS) == 10 and set(CATEGORY_IDS) == BASE_IDS | NOVEL_IDS
assert BASE_IDS.isdisjoint(NOVEL_IDS) and len(set(PROMPTS)) == 10
BASE_ID_TO_INDEX = {cat: i for i, cat in enumerate(sorted(BASE_IDS))}

def load_csv(name):
    with (PROTOCOL_DIR / name).open(newline="", encoding="utf-8") as stream:
        return list(csv.DictReader(stream))

train_rows = load_csv("train.csv")
val_rows = load_csv("val.csv")
test_rows = load_csv("test.csv")
safe_train_rows = load_csv("train_base_safe.csv")
assert len(train_rows) == spec["expected_images"]["train"]
assert len(val_rows) == spec["expected_images"]["val"]
assert len(test_rows) == spec["expected_images"]["test"]
train_keys = {r["image_relpath"] for r in train_rows}
assert train_keys.isdisjoint({r["image_relpath"] for r in val_rows})
assert train_keys.isdisjoint({r["image_relpath"] for r in test_rows})
assert {r["image_relpath"] for r in val_rows}.isdisjoint({r["image_relpath"] for r in test_rows})
assert {r["image_relpath"] for r in safe_train_rows}.issubset(train_keys)
assert all(int(r["num_novel_objects"]) == 0 for r in safe_train_rows)
safe_val_rows = [r for r in val_rows if int(r["num_novel_objects"]) == 0]
assert safe_val_rows, "No novel-free val images available for training validation."
eval_rows_full = val_rows if CFG.eval_split == "val" else test_rows

def chosen(rows, count):
    return sorted(rows, key=lambda r: r["image_relpath"]) if CFG.run_mode == "full" else sorted(
        random.Random(CFG.seed).sample(rows, min(count, len(rows))), key=lambda r: r["image_relpath"]
    )

train_selection = chosen(safe_train_rows, CFG.smoke_images)
val_selection = chosen(safe_val_rows, CFG.smoke_images)
eval_selection = chosen(eval_rows_full, CFG.smoke_images)

def local_file(relpath):
    path = (DATA_ROOT / relpath).resolve()
    if not path.is_relative_to(DATA_ROOT.resolve()):
        raise RuntimeError(f"Untrusted protocol relative path: {relpath}")
    return path

all_selected = {r["image_relpath"]: r for r in train_selection + val_selection + eval_selection}
for row in tqdm(all_selected.values(), desc="Check frozen image/annotation manifests"):
    for key, sha in (("image_relpath", "image_sha256"), ("annotation_relpath", "annotation_sha256")):
        path = local_file(row[key])
        if not path.is_file():
            raise FileNotFoundError(f"Missing source data: {path}")
        if CFG.verify_image_hashes and sha256_file(path) != row[sha]:
            raise RuntimeError(f"Source data modified since protocol registration: {path}")
print("Frozen Dataset ID:", PROTOCOL_ID, "| fingerprint:", FINGERPRINT)
print("Training/early-stopping/evaluation images:",
      len(train_selection), len(val_selection), len(eval_selection))
print("Base:", BASE_NAMES, "| Novel:", NOVEL_NAMES)
print("Full results?", CFG.run_mode == "full")


Check frozen image/annotation manifests:   0%|          | 0/3243 [00:00<?, ?it/s]

Frozen Dataset ID: 738dce9e1a5440dab34195418a997a20 | fingerprint: 158cb2c904016fa324c236b998393b1a94c176a4eabedb159c76bdeffc66c195
Training/early-stopping/evaluation images: 2695 119 548
Base: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck', 'motor'] | Novel: ['tricycle', 'awning-tricycle', 'bus']
Full results? True


## 2 · Define the unchanged training-view builder for each overlap

Whole base-safe images and the original annotation-integrity rule are unchanged. Training tiles use the requested overlap in both axes; validation remains whole-image only. Views are built sequentially by the training cell, each under its own overlap directory. The source protocol/images/annotations are never modified. A completed view can be reused within the same run.


In [5]:
def annotations_for_row(row):
    path = local_file(row["annotation_relpath"])
    records = []
    for lineno, raw in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):
        if not raw.strip():
            continue
        fields = [x.strip() for x in raw.split(",")]
        if len(fields) < 8:
            raise ValueError(f"{path}:{lineno}: expected 8 VisDrone columns")
        x, y, w, h = map(float, fields[:4])
        score, cid = int(fields[4]), int(fields[5])
        truncation, occlusion = int(fields[6]), int(fields[7])
        if cid not in {0, 11, *CATEGORY_IDS}:
            raise ValueError(f"{path}:{lineno}: unexpected category {cid}")
        if w <= 0 or h <= 0:
            continue
        records.append(dict(bbox=[x, y, w, h], score=score, category_id=cid,
                            truncation=truncation, occlusion=occlusion))
    return records


def training_sliding_positions(full, tile, overlap):
    if full <= tile:
        return [0]
    stride = max(1, round(tile * (1 - overlap)))
    points = list(range(0, full - tile + 1, stride))
    if points[-1] != full - tile:
        points.append(full - tile)
    return sorted(set(points))


def yolo_label_line(cid, x1, y1, x2, y2, width, height):
    return (f"{BASE_ID_TO_INDEX[cid]} {(x1+x2)/(2*width):.8f} "
            f"{(y1+y2)/(2*height):.8f} {(x2-x1)/width:.8f} {(y2-y1)/height:.8f}\n")


# This is an annotation-integrity rule, not a changed model hyperparameter.
# Whole images are retained, so large objects remain available even when a
# boundary tile is rejected.
MIN_TILE_VISIBLE_FRACTION = 0.50

def build_training_view(train_overlap):
    TRAIN_DATA_DIR = RUN_DIR / "train_data" / overlap_tag(train_overlap)
    marker = TRAIN_DATA_DIR / "training_view.json"
    signature = {"train_overlap": train_overlap, "protocol_fingerprint": FINGERPRINT,
                 "slice_height": CFG.slice_height, "slice_width": CFG.slice_width,
                 "train_images": [r["image_relpath"] for r in train_selection],
                 "val_images": [r["image_relpath"] for r in val_selection],
                 "min_visible_fraction": MIN_TILE_VISIBLE_FRACTION}
    if marker.is_file():
        completed = json.loads(marker.read_text(encoding="utf-8"))
        if completed["signature"] != signature:
            raise RuntimeError("Stored training-view signature differs from the frozen settings.")
        if not Path(completed["training_yaml"]).is_file():
            raise FileNotFoundError(completed["training_yaml"])
        print("Reuse generated training view:", TRAIN_DATA_DIR)
        return completed
    if TRAIN_DATA_DIR.exists():
        # Rebuild only an interrupted generated view, never the original dataset.
        assert TRAIN_DATA_DIR.is_relative_to(RUN_DIR / "train_data")
        shutil.rmtree(TRAIN_DATA_DIR)
    tile_count = 0
    whole_count = 0
    rejected_boundary_tiles = 0

    for split, rows in (("train", train_selection), ("val", val_selection)):
        image_out = TRAIN_DATA_DIR / split / "images"
        label_out = TRAIN_DATA_DIR / split / "labels"
        image_out.mkdir(parents=True)
        label_out.mkdir(parents=True)
        seen = set()

        for row in tqdm(rows, desc=f"Write {split} YOLO labels"):
            source = local_file(row["image_relpath"])
            if source.name in seen:
                raise RuntimeError(f"Duplicate image filename in selected {split} set: {source.name}")
            seen.add(source.name)
            annotations = annotations_for_row(row)

            with Image.open(source) as opened:
                image = opened.convert("RGB")
                W, H = image.size

                # Preserve the original whole-image view exactly as before.
                whole_target = image_out / source.name
                whole_target.symlink_to(source)
                whole_lines = []
                for ann in annotations:
                    cid = ann["category_id"]
                    if cid not in BASE_IDS:
                        if cid in NOVEL_IDS:
                            raise RuntimeError("A novel-class GT was found in a base-safe training/validation image.")
                        continue
                    x, y, w, h = ann["bbox"]
                    x1, y1 = max(0.0, min(float(W), x)), max(0.0, min(float(H), y))
                    x2, y2 = max(0.0, min(float(W), x + w)), max(0.0, min(float(H), y + h))
                    if x2 <= x1 or y2 <= y1:
                        continue
                    whole_lines.append(yolo_label_line(cid, x1, y1, x2, y2, W, H))
                (label_out / f"{source.stem}.txt").write_text("".join(whole_lines), encoding="utf-8")
                whole_count += 1

                # The original validation view is deliberately unchanged. Only
                # the training split receives additional sliced examples.
                if split != "train":
                    continue

                x_positions = training_sliding_positions(W, CFG.slice_width, train_overlap)
                y_positions = training_sliding_positions(H, CFG.slice_height, train_overlap)

                for top in y_positions:
                    for left in x_positions:
                        right = min(W, left + CFG.slice_width)
                        bottom = min(H, top + CFG.slice_height)
                        tile_w, tile_h = right - left, bottom - top
                        tile_lines = []
                        reject_tile = False

                        for ann in annotations:
                            cid = ann["category_id"]
                            if cid not in BASE_IDS:
                                continue
                            x, y, w, h = ann["bbox"]
                            box_x1 = max(0.0, min(float(W), x))
                            box_y1 = max(0.0, min(float(H), y))
                            box_x2 = max(0.0, min(float(W), x + w))
                            box_y2 = max(0.0, min(float(H), y + h))
                            box_area = max(0.0, box_x2 - box_x1) * max(0.0, box_y2 - box_y1)
                            if box_area <= 0:
                                continue

                            inter_x1 = max(box_x1, float(left))
                            inter_y1 = max(box_y1, float(top))
                            inter_x2 = min(box_x2, float(right))
                            inter_y2 = min(box_y2, float(bottom))
                            inter_area = max(0.0, inter_x2 - inter_x1) * max(0.0, inter_y2 - inter_y1)
                            if inter_area <= 0:
                                continue

                            visible_fraction = inter_area / box_area
                            if visible_fraction < MIN_TILE_VISIBLE_FRACTION:
                                reject_tile = True
                                break

                            tile_lines.append(yolo_label_line(
                                cid,
                                inter_x1 - left, inter_y1 - top,
                                inter_x2 - left, inter_y2 - top,
                                tile_w, tile_h,
                            ))

                        # Whole images already provide background context. Added
                        # slices are restricted to positive, non-severely-cut views
                        # so empty tiles do not overwhelm the original dataset.
                        if reject_tile:
                            rejected_boundary_tiles += 1
                            continue
                        if not tile_lines:
                            continue

                        tile_stem = f"tile__{source.stem}__y{top:05d}_x{left:05d}"
                        tile_path = image_out / f"{tile_stem}.jpg"
                        image.crop((left, top, right, bottom)).save(
                            tile_path, format="JPEG", quality=95, subsampling=0
                        )
                        (label_out / f"{tile_stem}.txt").write_text(
                            "".join(tile_lines), encoding="utf-8"
                        )
                        tile_count += 1


    # Base names in ascending official-ID order: model classes 0..6 during training.
    train_yaml = TRAIN_DATA_DIR / "base_only.yaml"
    train_yaml.write_text(
        f"path: {TRAIN_DATA_DIR.as_posix()}\ntrain: train/images\nval: val/images\n"
        + "names:\n" + "".join(f"  {i}: {cats['official_category_id_to_name'][str(cid)]}\n"
                                  for cid, i in BASE_ID_TO_INDEX.items()),
        encoding="utf-8",
    )
    print("YOLO training YAML:", train_yaml)
    print("Whole training/validation images retained:", whole_count)
    print("Additional positive training tiles:", tile_count)
    print("Rejected boundary tiles:", rejected_boundary_tiles)

    view = {"signature": signature, "training_yaml": str(train_yaml),
            "training_data_dir": str(TRAIN_DATA_DIR), "train_overlap": train_overlap,
            "whole_train_and_val_images": whole_count, "positive_training_tiles": tile_count,
            "rejected_boundary_tiles": rejected_boundary_tiles}
    marker.write_text(json.dumps(view, indent=2) + "\n", encoding="utf-8")
    return view


## 3 · Create a NEW ClearML experiment project and local output directories

No bucket, AWS credentials, or S3 configuration is needed. This notebook stores the large weights and complete prediction files under the 50 GB SageMaker space. Each run gets a unique ID. Ultralytics automatic ClearML checkpoint upload is disabled to avoid consuming the ClearML file quota. Do not execute this notebook with an already-active unrelated ClearML task. Monitor free disk space when running more models; `!df -h ~` shows the available space.


In [6]:
from ultralytics import YOLOWorld, __version__ as ULTRALYTICS_VERSION, settings

# Set YOLO_CONFIG_DIR *before* importing ultralytics. Disabling its auto-ClearML
# callback prevents duplicate tasks and uncontrolled checkpoint uploads. This
# per-run settings directory does not modify other notebooks' YOLO settings.
settings.update({"clearml": False})
if settings["clearml"]:
    raise RuntimeError("Disable Ultralytics ClearML auto-upload before training.")
if Task.current_task() is not None:
    raise RuntimeError("An unrelated ClearML task is active. Restart the kernel before this notebook.")

task = Task.init(
    project_name=CFG.clearml_project,
    task_name=f"{CFG.task_name_prefix} | YOLO-World | {CFG.run_mode} | {CFG.eval_split} | {RUN_ID}",
    reuse_last_task_id=False,
    auto_connect_frameworks=False,
)
logger = task.get_logger()
cfg_logged = asdict(CFG)
task.connect(cfg_logged, name="experiment_config")
task.connect({"protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
              "local_weights_dir": str(WEIGHTS_DIR),
              "local_results_dir": str(RESULTS_DIR),
              "run_id": RUN_ID, "ultralytics_version": ULTRALYTICS_VERSION,
              "torch_version": torch.__version__, "python": platform.python_version()}, name="provenance")
print("ClearML task:", task.id)

def record_local_file(path):
    path = Path(path).resolve()
    if not path.is_file() or not (path.is_relative_to(WEIGHTS_DIR) or path.is_relative_to(RESULTS_DIR)):
        raise ValueError(f"Expected a stored file under weights/ or results/: {path}")
    return {"local_path": str(path), "file_uri": path.as_uri(),
            "bytes": path.stat().st_size, "sha256": sha256_file(path)}

def store_local_file(source_path, section, filename):
    source_path = Path(source_path).resolve()
    if not source_path.is_file() or section not in {"weights", "results"}:
        raise ValueError("Only existing source files and weights/results sections are allowed.")
    if Path(filename).name != filename or filename in {"", ".", ".."}:
        raise ValueError("Use a plain filename with no path components.")
    destination = (WEIGHTS_DIR if section == "weights" else RESULTS_DIR) / filename
    if destination.exists():
        if not destination.samefile(source_path):
            raise FileExistsError(f"Refusing to overwrite an earlier output: {destination}")
    else:
        # Hard-link where both paths share the EBS volume: avoids using disk twice.
        # If the source is on another filesystem, copy instead.
        temporary = destination.with_name(f".{filename}.{uuid.uuid4().hex}.tmp")
        try:
            try:
                os.link(source_path, temporary)
            except OSError:
                shutil.copy2(source_path, temporary)
            if sha256_file(temporary) != sha256_file(source_path):
                raise RuntimeError(f"Local copy did not pass SHA-256 verification: {source_path}")
            os.replace(temporary, destination)
        finally:
            temporary.unlink(missing_ok=True)
    return record_local_file(destination)

LOCAL_INDEX = {"weights": {}, "results": {}}


WARNING ⚠️ torchvision==0.24 is incompatible with torch==2.8.
Run 'pip install torchvision==0.23' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yolo_world_v2/20261008T163054Z_73692224/ultralytics_config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
ClearML Task: created new task id=e4969168f378468ea4ee70d71787e30e
ClearML results page: https://app.clear.ml/projects/90b0c1c7704844ccaae03dcd11c5e6b8/tasks/e4969168f378468ea4ee70d71787e30e/output/log
ClearML task: e4969168f378468ea4ee70d71787e30e


## 4 · Five independent fine-tuning runs, varying only training tile overlap

Each run starts from the same pretrained weights and the same seed. All original train() arguments are retained, apart from the overlap-specific data and output paths. Best and last checkpoints, training histories and tile counts are saved separately for 10%, 20%, 30%, 40%, 50%. The original whole-image base-safe validation still selects best.pt. Models are released between runs to avoid keeping five models in GPU memory.


In [ ]:
def set_frozen_prompts(model):
    model.set_classes(PROMPTS)
    active = [model.names[k] for k in sorted(model.names)]
    if active[:len(PROMPTS)] != PROMPTS:
        raise RuntimeError(f"Prompt order mismatch: expected {PROMPTS!r}, got {active!r}")
    return model

baseline = YOLOWorld(CFG.yolo_checkpoint)
pretrained_path = Path(getattr(baseline, "ckpt_path", CFG.yolo_checkpoint))
if not pretrained_path.is_file():
    pretrained_path = Path(CFG.yolo_checkpoint)
if not pretrained_path.is_file():
    raise FileNotFoundError("Ultralytics pretrained checkpoint not found after model loading.")
LOCAL_INDEX["weights"]["pretrained.pt"] = store_local_file(pretrained_path, "weights", "pretrained.pt")
baseline = set_frozen_prompts(baseline)

# The same pristine pretrained checkpoint is reused for all five initializations.
baseline.to("cpu")
del baseline
gc.collect()
torch.cuda.empty_cache()

TRAINING_RUNS = {}
training_view_rows = []
for train_overlap in CFG.train_overlap_ratios:
    training_key = overlap_tag(train_overlap)
    print(f"\n=== Training overlap {train_overlap:.0%} ===")
    view = build_training_view(train_overlap)
    train_yaml = Path(view["training_yaml"])
    training_view_rows.append({key: value for key, value in view.items() if key != "signature"})
    pd.DataFrame(training_view_rows).to_csv(RESULTS_DIR / "training_views.csv", index=False)
    random.seed(CFG.seed)
    np.random.seed(CFG.seed)
    torch.manual_seed(CFG.seed)
    torch.cuda.manual_seed_all(CFG.seed)
    # Create a fresh model for fine-tuning; no zero-shot prompt changes bleed into training.
    trainer = YOLOWorld(LOCAL_INDEX["weights"]["pretrained.pt"]["local_path"])
    training_results = trainer.train(
        data=str(train_yaml),
        epochs=CFG.smoke_epochs if CFG.run_mode == "smoke" else CFG.epochs,
        imgsz=CFG.yolo_image_size,
        batch=CFG.batch,
        optimizer="AdamW",
        lr0=0.0002,
        device=0,
        seed=CFG.seed,
        deterministic=True,
        project=str(RUN_DIR / "training"),
        name=f"base_safe_{training_key}",
        exist_ok=False,
        workers=4,
        save=True,
        plots=False,
    )
    checkpoint_dir = Path(trainer.trainer.save_dir) / "weights"
    best_path = checkpoint_dir / "best.pt"
    last_path = checkpoint_dir / "last.pt"
    for name, path in (("best.pt", best_path), ("last.pt", last_path)):
        if not path.is_file():
            raise FileNotFoundError(f"Training did not save expected checkpoint: {path}")
        LOCAL_INDEX["weights"][f"train_{training_key}_{name}"] = store_local_file(path, "weights", f"train_{training_key}_{name}")
    print("Checkpoints verified locally:", LOCAL_INDEX["weights"])

    # Register local file URIs as metadata, without uploading model bytes to ClearML.
    # They are accessible only from an environment with this SageMaker directory mounted.
    from clearml import OutputModel
    for checkpoint_name in ("best.pt", "last.pt"):
        registered = OutputModel(task=task, name=f"YOLO-World-v2 {training_key} {checkpoint_name} | {RUN_ID}",
                                 framework="PyTorch", label_enumeration={name: i for i, name in enumerate(BASE_NAMES)},
                                 comment=f"VisDrone base-safe protocol {PROTOCOL_ID}; see local checkpoint and frozen prompts")
        registered.update_weights(register_uri=LOCAL_INDEX["weights"][f"train_{training_key}_{checkpoint_name}"]["file_uri"],
                                  async_enable=False)

    training_csv = Path(trainer.trainer.save_dir) / "results.csv"
    if training_csv.is_file():
        LOCAL_INDEX["results"][f"training_history_{training_key}.csv"] = store_local_file(training_csv, "results", f"training_history_{training_key}.csv")
        history = pd.read_csv(training_csv)
        for column in history.columns:
            if column.strip().startswith(("train/", "val/", "metrics/")):
                for epoch, value in enumerate(history[column].values):
                    if pd.notna(value):
                        logger.report_scalar(f"Training history {training_key}", column.strip(), float(value), epoch)

    best_info = LOCAL_INDEX["weights"][f"train_{training_key}_best.pt"]
    last_info = LOCAL_INDEX["weights"][f"train_{training_key}_last.pt"]
    TRAINING_RUNS[training_key] = {"train_overlap": train_overlap,
                                  "best_path": best_info["local_path"],
                                  "last_path": last_info["local_path"],
                                  "best_checkpoint_sha256": best_info["sha256"],
                                  "training_yaml": str(train_yaml),
                                  "training_save_dir": str(trainer.trainer.save_dir)}
    (RESULTS_DIR / "training_runs.json").write_text(
        json.dumps(TRAINING_RUNS, indent=2) + "\n", encoding="utf-8")
    trainer.to("cpu")
    del trainer, training_results
    gc.collect()
    torch.cuda.empty_cache()

assert len(TRAINING_RUNS) == 5
print("Five independent overlap checkpoints saved:", WEIGHTS_DIR)


E0000 00:00:1791477093.481993    6263 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791477093.490099    6263 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1791477093.509418    6263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791477093.509445    6263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791477093.509448    6263 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1791477093.509451    6263 computation_placer.cc:177] computation placer already registered. Please check linka


=== Training overlap 10% ===


Write train YOLO labels:   0%|          | 0/2695 [00:00<?, ?it/s]

Write val YOLO labels:   0%|          | 0/119 [00:00<?, ?it/s]

YOLO training YAML: /home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yolo_world_v2/20261008T163054Z_73692224/train_data/overlap_10/base_only.yaml
Whole training/validation images retained: 2814
Additional positive training tiles: 16162
Rejected boundary tiles: 27570
Ultralytics 8.4.174 🚀 Python-3.12.14 torch-2.8.0 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/sagemaker-user/SageMaker_VisDrone_OVD_2026/runs/yolo_world_v2/20261008T163054Z_73692224/train_data/overlap_10/base_only.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, e

## 5 · Exactly 25 sliced inference conditions on the same frozen evaluation images

For each of the five fine-tuned checkpoints, test inference overlap 10%, 20%, 30%, 40%, 50% in both axes. The original RGB/BGR handling, per-tile NMS, class-aware global NMS, confidence floors, input size and max detections are unchanged. Warmup remains uncounted; GPU synchronization and the original pipeline latency measurement are retained. Save complete outputs after every condition, and load only one checkpoint at a time. The original whole-frame inference helper is retained without adding whole-frame/zero-shot tests to the requested grid.


In [ ]:
def sliding_positions(full, tile, overlap):
    if full <= tile:
        return [0]
    stride = max(1, round(tile * (1 - overlap)))
    points = list(range(0, full - tile + 1, stride))
    if points[-1] != full - tile:
        points.append(full - tile)
    return sorted(set(points))

def bbox_iou_xyxy(a, b):
    x1, y1 = max(a[0], b[0]), max(a[1], b[1])
    x2, y2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0., x2 - x1) * max(0., y2 - y1)
    area_a = max(0., a[2] - a[0]) * max(0., a[3] - a[1])
    area_b = max(0., b[2] - b[0]) * max(0., b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.

def classwise_nms(predictions):
    chosen = []
    by_class = defaultdict(list)
    for p in predictions:
        by_class[p["category_id"]].append(p)
    for group in by_class.values():
        for p in sorted(group, key=lambda r: -r["score"]):
            if all(bbox_iou_xyxy(p["xyxy"], kept["xyxy"]) < CFG.global_nms_iou for kept in chosen
                   if kept["category_id"] == p["category_id"]):
                chosen.append(p)
    return sorted(chosen, key=lambda r: -r["score"])[:CFG.max_detections_per_image]

def predict_region(model, rgb_array, off_x=0, off_y=0):
    # PIL arrays are RGB; Ultralytics ndarray inputs are interpreted as BGR.
    bgr = np.ascontiguousarray(rgb_array[:, :, ::-1])
    result = model.predict(source=bgr, imgsz=CFG.yolo_image_size, conf=CFG.yolo_extraction_threshold,
                           iou=CFG.yolo_local_nms_iou, max_det=CFG.max_detections_per_image,
                           device=0, verbose=False)[0]
    boxes = result.boxes
    if boxes is None or len(boxes) == 0:
        return []
    coords = boxes.xyxy.detach().cpu().numpy()
    classes = boxes.cls.detach().cpu().numpy().astype(int)
    scores = boxes.conf.detach().cpu().numpy()
    predictions = []
    for b, cls, conf in zip(coords, classes, scores):
        if cls >= len(CATEGORY_IDS):
            continue
        predictions.append({"category_id": CATEGORY_IDS[cls], "score": float(conf),
                            "xyxy": [float(b[0]+off_x), float(b[1]+off_y),
                                     float(b[2]+off_x), float(b[3]+off_y)]})
    return predictions

def run_predict(model, rgb_array, mode, overlap_ratio=None):
    H, W = rgb_array.shape[:2]
    if mode == "whole":
        return predict_region(model, rgb_array), 1
    width_overlap = CFG.overlap_width_ratio if overlap_ratio is None else overlap_ratio
    height_overlap = CFG.overlap_height_ratio if overlap_ratio is None else overlap_ratio
    xs = sliding_positions(W, CFG.slice_width, width_overlap)
    ys = sliding_positions(H, CFG.slice_height, height_overlap)
    raw = []
    if CFG.include_whole_prediction_in_sliced:
        raw.extend(predict_region(model, rgb_array))
    for y in ys:
        for x in xs:
            tile = rgb_array[y:min(y + CFG.slice_height, H), x:min(x + CFG.slice_width, W)]
            raw.extend(predict_region(model, tile, x, y))
    return classwise_nms(raw), len(xs) * len(ys)

# Exactly 5 independently trained checkpoints x 5 sliced inference overlaps.
# Whole-frame and zero-shot helpers are retained, but are not extra grid tests.
CONDITIONS = [
    {"condition": f"train_{overlap_tag(train_overlap)}__eval_{overlap_tag(eval_overlap)}",
     "training_key": overlap_tag(train_overlap),
     "train_overlap": train_overlap, "eval_overlap": eval_overlap}
    for train_overlap in CFG.train_overlap_ratios
    for eval_overlap in CFG.eval_overlap_ratios
]
assert len(TRAINING_RUNS) == 5 and len(CONDITIONS) == 25
assert len({item["condition"] for item in CONDITIONS}) == 25

def save_condition_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    os.replace(temporary, path)

def condition_paths(condition):
    name = condition["condition"]
    return (RESULTS_DIR / f"raw_predictions_{name}.json",
            RESULTS_DIR / f"latency_{name}.json",
            RESULTS_DIR / f"completed_{name}.json")

def condition_signature(condition):
    training = TRAINING_RUNS[condition["training_key"]]
    signature = {"condition": condition, "config": asdict(CFG),
            "protocol_fingerprint": FINGERPRINT,
            "checkpoint_sha256": training["best_checkpoint_sha256"],
            "eval_image_relpaths": [r["image_relpath"] for r in eval_selection],
            "prompts": PROMPTS, "ultralytics_version": ULTRALYTICS_VERSION}
    # Match JSON-loaded lists against JSON-normalized tuple config values.
    return json.loads(json.dumps(signature))

with Image.open(local_file(eval_selection[0]["image_relpath"])) as first:
    warmup_image = np.asarray(first.convert("RGB"))

for training_key, training in TRAINING_RUNS.items():
    # Load only one of the five checkpoints onto the GPU at a time.
    model = set_frozen_prompts(YOLOWorld(training["best_path"]))
    try:
        for condition in [c for c in CONDITIONS if c["training_key"] == training_key]:
            name = condition["condition"]
            prediction_path, latency_path, marker_path = condition_paths(condition)
            signature = condition_signature(condition)
            if CFG.skip_completed_conditions and all(p.is_file() for p in (prediction_path, latency_path, marker_path)):
                saved = json.loads(marker_path.read_text(encoding="utf-8"))
                if (saved["signature"] == signature
                    and saved["prediction_sha256"] == sha256_file(prediction_path)
                    and saved["latency_sha256"] == sha256_file(latency_path)):
                    print("Reuse completed inference:", name)
                    continue
            with torch.inference_mode():
                run_predict(model, warmup_image, "sliced", condition["eval_overlap"])
            outputs, timings = {}, []
            for row in tqdm(eval_selection, desc=f"Inference {name}"):
                with Image.open(local_file(row["image_relpath"])) as image:
                    rgb = np.asarray(image.convert("RGB"))
                torch.cuda.synchronize()
                start = time.perf_counter()
                with torch.inference_mode():
                    predictions, n_tiles = run_predict(model, rgb, "sliced", condition["eval_overlap"])
                torch.cuda.synchronize()
                millis = 1000 * (time.perf_counter() - start)
                outputs[row["image_relpath"]] = predictions
                timings.append({"image_relpath": row["image_relpath"], "pipeline_ms": millis, "tiles": n_tiles})
            save_condition_json(prediction_path, outputs)
            save_condition_json(latency_path, timings)
            # A completion marker is written only after both full files are saved.
            save_condition_json(marker_path, {"signature": signature,
                                "prediction_sha256": sha256_file(prediction_path),
                                "latency_sha256": sha256_file(latency_path)})
            print(name, "mean latency ms:", round(float(np.mean([x["pipeline_ms"] for x in timings])), 2))
            del outputs, timings, rgb, predictions
    finally:
        model.to("cpu")
        del model
        gc.collect()
        torch.cuda.empty_cache()

print("All 25 inference conditions saved:", RESULTS_DIR)


## 6 · Original COCO/operating metrics plus GT counts and image comparisons

Keep AP@[.50:.95], AP50, AR500, per-class/base/novel metrics, custom size bins and occlusion recall exactly as before. Add object counts overall, per class and per image: GT, predictions at the operating threshold, correctly detected objects, missed objects and false positives. Same-category one-to-one matching uses the existing confidence 0.25 and IoU 0.5, so correctly detected objects / GT equals operating recall. Each condition saves every evaluation image as a GT/prediction side-by-side JPEG; this image writing does not affect timed inference. GT ignore-ID handling remains unchanged.


In [ ]:
from contextlib import redirect_stdout
from io import StringIO

def size_bin(area):
    if area < 16**2:
        return "tiny"
    if area < 32**2:
        return "small"
    if area < 96**2:
        return "medium"
    return "large"

gt_records = {}
coco_images = []
coco_anns = []
for img_id, row in enumerate(eval_selection, 1):
    path = local_file(row["image_relpath"])
    with Image.open(path) as image:
        W, H = image.size
    coco_images.append({"id": img_id, "width": W, "height": H, "file_name": row["image_relpath"]})
    gt_records[row["image_relpath"]] = []
    for ann in annotations_for_row(row):
        if ann["category_id"] not in CATEGORY_IDS:
            continue
        x, y, w, h = ann["bbox"]
        x1, y1, x2, y2 = max(0., x), max(0., y), min(float(W), x+w), min(float(H), y+h)
        if x2 <= x1 or y2 <= y1:
            continue
        ground = {"id": len(coco_anns)+1, "image_id": img_id, "category_id": ann["category_id"],
                  "bbox": [x1, y1, x2-x1, y2-y1], "xyxy": [x1, y1, x2, y2],
                  "area": (x2-x1)*(y2-y1), "size_bin": size_bin((x2-x1)*(y2-y1)),
                  "occlusion": ann["occlusion"]}
        gt_records[row["image_relpath"]].append(ground)
        coco_anns.append({key: ground[key] for key in ("id", "image_id", "category_id", "bbox", "area")}
                         | {"iscrowd": 0})
assert coco_anns, "No evaluation ground truth."
coco_gt = COCO()
coco_gt.dataset = {"info": {"description": "VisDrone original fixed-split evaluation"},
                   "licenses": [], "images": coco_images,
                   "categories": [{"id": cid, "name": name} for cid, name in zip(CATEGORY_IDS, CATEGORY_NAMES)],
                   "annotations": coco_anns}
coco_gt.createIndex()

def valid_mean(values):
    valid = np.asarray(values)[np.asarray(values) > -1]
    return float(valid.mean()) if valid.size else None

def measure_coco(outputs):
    predictions = []
    for img_id, row in enumerate(eval_selection, 1):
        for p in outputs[row["image_relpath"]]:
            x1, y1, x2, y2 = p["xyxy"]
            predictions.append({"image_id": img_id, "category_id": p["category_id"],
                                "bbox": [x1, y1, max(0., x2-x1), max(0., y2-y1)], "score": p["score"]})
    if predictions:
        with redirect_stdout(StringIO()):
            coco_dt = coco_gt.loadRes(predictions)
    else:
        coco_dt = COCO()
        coco_dt.dataset = {"images": coco_images, "categories": coco_gt.dataset["categories"],
                           "annotations": [], "info": {}}
        coco_dt.createIndex()
    ev = COCOeval(coco_gt, coco_dt, "bbox")
    ev.params.catIds = CATEGORY_IDS
    ev.params.imgIds = [x["id"] for x in coco_images]
    ev.params.maxDets = [1, 10, CFG.max_detections_per_image]
    ev.params.areaRng = [[0, 1e12], [0, 16**2], [16**2, 32**2], [32**2, 96**2], [96**2, 1e12]]
    ev.params.areaRngLbl = ["all", "tiny", "small", "medium", "large"]
    with redirect_stdout(StringIO()):
        ev.evaluate()
        ev.accumulate()
    precision = ev.eval["precision"]  # IoU, recall step, class, area, maxDets
    recall = ev.eval["recall"]        # IoU, class, area, maxDets
    gt_counts = Counter(g["category_id"] for arr in gt_records.values() for g in arr)
    metrics = {}
    groups = {"all": CATEGORY_IDS, "base": sorted(BASE_IDS), "novel": sorted(NOVEL_IDS)}
    for group, ids in groups.items():
        indexes = [CATEGORY_IDS.index(cid) for cid in ids if gt_counts[cid] > 0]
        metrics[f"{group}_AP"] = valid_mean(precision[:, :, indexes, 0, -1])
        metrics[f"{group}_AP50"] = valid_mean(precision[0, :, indexes, 0, -1])
        metrics[f"{group}_AR500"] = valid_mean(recall[:, indexes, 0, -1])
    per_class = []
    for j, (cid, label) in enumerate(zip(CATEGORY_IDS, CATEGORY_NAMES)):
        per_class.append({"category_id": cid, "category": label, "split_group": "base" if cid in BASE_IDS else "novel",
                          "gt_count": gt_counts[cid], "AP": valid_mean(precision[:, :, j, 0, -1]),
                          "AP50": valid_mean(precision[0, :, j, 0, -1]),
                          "AR500": valid_mean(recall[:, j, 0, -1])})
    sizes = []
    for idx, bin_name in enumerate(("tiny", "small", "medium", "large"), 1):
        sizes.append({"size_bin": bin_name,
                      "gt_count": sum(g["size_bin"] == bin_name for arr in gt_records.values() for g in arr),
                      "AP": valid_mean(precision[:, :, :, idx, -1]),
                      "AP50": valid_mean(precision[0, :, :, idx, -1]),
                      "AR500": valid_mean(recall[:, :, idx, -1])})
    return metrics, per_class, sizes, predictions

def match_image_operating(gts, predictions):
    """Same score-ordered, same-category, one-to-one matching as before."""
    used = set()
    matches = []
    for pred in sorted((p for p in predictions if p["score"] >= CFG.operating_threshold_yolo),
                       key=lambda p: -p["score"]):
        candidates = [(bbox_iou_xyxy(pred["xyxy"], g["xyxy"]), g)
                      for g in gts if g["category_id"] == pred["category_id"] and g["id"] not in used]
        ground = None
        if candidates:
            overlap, candidate = max(candidates, key=lambda x: x[0])
            if overlap >= CFG.evaluation_iou:
                ground = candidate
                used.add(ground["id"])
        matches.append((pred, ground))
    tp = len(used)
    fp = len(matches) - tp
    fn = len(gts) - tp
    counts = {"ground_truth_objects": len(gts), "predicted_objects": len(matches),
              "correctly_detected_objects": tp, "missed_objects": fn,
              "false_positive_objects": fp,
              "object_detection_rate": tp / len(gts) if gts else None}
    return matches, counts

def match_operating(outputs):
    counts = Counter()
    attributes = defaultdict(lambda: [0, 0])
    by_class = {cid: Counter() for cid in CATEGORY_IDS}
    image_counts = []
    for row in eval_selection:
        key = row["image_relpath"]
        gts = gt_records[key]
        matches, image_count = match_image_operating(gts, outputs[key])
        image_counts.append({"image_relpath": key, **image_count})
        for g in gts:
            attributes[("size_bin", g["size_bin"])][0] += 1
            occ = {0: "none", 1: "partial", 2: "heavy"}.get(g["occlusion"], f"other_{g['occlusion']}")
            attributes[("occlusion", occ)][0] += 1
            by_class[g["category_id"]]["ground_truth_objects"] += 1
        for pred, ground in matches:
            by_class[pred["category_id"]]["predicted_objects"] += 1
            if ground is not None:
                counts["TP"] += 1
                by_class[pred["category_id"]]["correctly_detected_objects"] += 1
                attributes[("size_bin", ground["size_bin"])][1] += 1
                occ = {0: "none", 1: "partial", 2: "heavy"}.get(ground["occlusion"], f"other_{ground['occlusion']}")
                attributes[("occlusion", occ)][1] += 1
            else:
                counts["FP"] += 1
                by_class[pred["category_id"]]["false_positive_objects"] += 1
        counts["FN"] += image_count["missed_objects"]
    tp, fp, fn = (counts[x] for x in ("TP", "FP", "FN"))
    P = tp / (tp + fp) if tp + fp else 0.
    R = tp / (tp + fn) if tp + fn else 0.
    F1 = 2*P*R/(P+R) if P+R else 0.
    metrics = {"TP": tp, "FP": fp, "FN": fn, "Precision@0.5": P, "Recall@0.5": R, "F1@0.5": F1}
    class_counts = {}
    for cid, values in by_class.items():
        gt_count = values["ground_truth_objects"]
        hits = values["correctly_detected_objects"]
        class_counts[cid] = {"ground_truth_objects": gt_count,
                             "predicted_objects": values["predicted_objects"],
                             "correctly_detected_objects": hits,
                             "missed_objects": gt_count - hits,
                             "false_positive_objects": values["false_positive_objects"],
                             "object_detection_rate": hits / gt_count if gt_count else None}
    if CFG.report_ground_truth_objects:
        metrics.update({"ground_truth_objects": tp + fn, "predicted_objects": tp + fp,
                        "correctly_detected_objects": tp, "missed_objects": fn,
                        "false_positive_objects": fp, "object_detection_rate": R})
    robustness = [
        {"attribute": attr, "level": level, "gt_count": num, "Recall@0.5": hit/num if num else None}
        for (attr, level), (num, hit) in sorted(attributes.items())
    ]
    return metrics, robustness, image_counts, class_counts

from PIL import ImageDraw, ImageFont

CATEGORY_LABELS = dict(zip(CATEGORY_IDS, CATEGORY_NAMES))
_palette = ["#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231",
            "#911eb4", "#46f0f0", "#f032e6", "#bcf60c", "#fabebe"]
CATEGORY_COLORS = dict(zip(CATEGORY_IDS, _palette))

def comparison_font(size):
    try:
        return ImageFont.truetype("DejaVuSans.ttf", size)
    except OSError:
        return ImageFont.load_default()

def draw_box_labels(image, objects, predictions=False):
    draw = ImageDraw.Draw(image)
    font = comparison_font(12)
    for obj in objects:
        x1, y1, x2, y2 = obj["xyxy"]
        x1, y1 = max(0., x1), max(0., y1)
        x2, y2 = min(image.width - 1., x2), min(image.height - 1., y2)
        if x2 <= x1 or y2 <= y1:
            continue
        color = CATEGORY_COLORS[obj["category_id"]]
        label = CATEGORY_LABELS[obj["category_id"]]
        if predictions:
            label += f" {obj['score']:.2f}"
        draw.rectangle((x1, y1, x2, y2), outline=color, width=2)
        bounds = draw.textbbox((0, 0), label, font=font)
        text_w, text_h = bounds[2] - bounds[0], bounds[3] - bounds[1]
        tx = min(x1, max(0., image.width - text_w - 4))
        ty = max(0., y1 - text_h - 5)
        draw.rectangle((tx, ty, tx + text_w + 4, ty + text_h + 4), fill=color)
        draw.text((tx + 2 - bounds[0], ty + 2 - bounds[1]), label, fill="black", font=font)
    return image

def save_comparison_image(row, predictions, condition, object_counts):
    # No detector call here: render the exact merged predictions already scored.
    with Image.open(local_file(row["image_relpath"])) as opened:
        rgb = opened.convert("RGB")
    ground = draw_box_labels(rgb.copy(), gt_records[row["image_relpath"]])
    visible_predictions = [p for p in predictions if p["score"] >= CFG.operating_threshold_yolo]
    detected = draw_box_labels(rgb.copy(), visible_predictions, predictions=True)
    header_height = 88
    canvas = Image.new("RGB", (2 * rgb.width, rgb.height + header_height), "white")
    canvas.paste(ground, (0, header_height))
    canvas.paste(detected, (rgb.width, header_height))
    draw = ImageDraw.Draw(canvas)
    title_font, body_font = comparison_font(18), comparison_font(14)
    train_pct, eval_pct = int(round(condition["train_overlap"] * 100)), int(round(condition["eval_overlap"] * 100))
    draw.text((12, 8), f"Train overlap {train_pct}% | Inference overlap {eval_pct}% | {CFG.eval_split}",
              fill="black", font=title_font)
    draw.text((12, 38), f"Ground truth | objects={object_counts['ground_truth_objects']}", fill="black", font=body_font)
    draw.text((rgb.width + 12, 38),
              f"Predictions | objects={object_counts['predicted_objects']} | conf >= {CFG.operating_threshold_yolo:g}",
              fill="black", font=body_font)
    draw.text((rgb.width + 12, 61),
              f"TP={object_counts['correctly_detected_objects']}  FP={object_counts['false_positive_objects']}  "
              f"FN={object_counts['missed_objects']} | same class, IoU >= {CFG.evaluation_iou:g}",
              fill="black", font=body_font)
    draw.line((rgb.width, 32, rgb.width, canvas.height), fill="white", width=4)
    # Preserve the protocol-relative subdirectory to prevent filename collisions.
    destination = (RESULTS_DIR / "comparisons" / condition["condition"] /
                   Path(row["image_relpath"]).with_suffix(".jpg"))
    destination.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(destination, format="JPEG", quality=95, subsampling=0)
    return {"condition": condition["condition"], "train_overlap": condition["train_overlap"],
            "eval_overlap": condition["eval_overlap"], "image_relpath": row["image_relpath"],
            "comparison_relpath": str(destination.relative_to(RESULTS_DIR)), **object_counts}


summary_rows, class_rows, size_rows, robustness_rows = [], [], [], []
image_count_rows, comparison_rows = [], []
for condition in CONDITIONS:
    name = condition["condition"]
    prediction_path, latency_path, marker_path = condition_paths(condition)
    outputs = json.loads(prediction_path.read_text(encoding="utf-8"))
    timings = json.loads(latency_path.read_text(encoding="utf-8"))
    expected_keys = {r["image_relpath"] for r in eval_selection}
    if set(outputs) != expected_keys or {t["image_relpath"] for t in timings} != expected_keys:
        raise RuntimeError(f"Prediction files do not cover the frozen evaluation split: {name}")
    metrics, classes, sizes, preds = measure_coco(outputs)
    operating, robustness, image_counts, class_counts = match_operating(outputs)
    assert operating["TP"] + operating["FN"] == len(coco_anns)
    times = np.array([x["pipeline_ms"] for x in timings])
    condition_meta = {"condition": name, "train_overlap": condition["train_overlap"],
                      "eval_overlap": condition["eval_overlap"]}
    summary_rows.append({**condition_meta, "protocol_id": PROTOCOL_ID, "eval_split": CFG.eval_split,
                         "run_mode": CFG.run_mode, "images": len(eval_selection),
                         "threshold": CFG.operating_threshold_yolo, **metrics, **operating,
                         "mean_pipeline_ms": float(times.mean()), "median_pipeline_ms": float(np.median(times)),
                         "p95_pipeline_ms": float(np.percentile(times, 95)),
                         "fps_from_pipeline_mean": 1000/float(times.mean()),
                         "mean_tiles": float(np.mean([x["tiles"] for x in timings]))})
    class_rows.extend({**condition_meta, **item,
                       **(class_counts[item["category_id"]] if CFG.report_ground_truth_objects else {})}
                      for item in classes)
    size_rows.extend({**condition_meta, **item} for item in sizes)
    robustness_rows.extend({**condition_meta, **item} for item in robustness)
    if CFG.report_ground_truth_objects:
        image_count_rows.extend({**condition_meta, **item} for item in image_counts)
        pd.DataFrame(image_counts).to_csv(RESULTS_DIR / f"object_counts_{name}.csv", index=False)
    # Complete outputs are saved after each condition; do not retain 25 prediction sets in RAM.
    save_condition_json(RESULTS_DIR / f"predictions_{name}.json", preds)
    if CFG.save_comparison_images:
        count_by_image = {item["image_relpath"]: item for item in image_counts}
        for row in tqdm(eval_selection, desc=f"GT/prediction comparisons {name}"):
            counts = {key: value for key, value in count_by_image[row["image_relpath"]].items()
                      if key != "image_relpath"}
            comparison_rows.append(save_comparison_image(row, outputs[row["image_relpath"]], condition, counts))
    summary_df = pd.DataFrame(summary_rows)
    class_df = pd.DataFrame(class_rows)
    size_df = pd.DataFrame(size_rows)
    robust_df = pd.DataFrame(robustness_rows)
    object_count_df = pd.DataFrame(image_count_rows)
    comparisons_df = pd.DataFrame(comparison_rows)
    # Preserve progress even if a later condition is interrupted.
    for filename, frame in (("summary.csv", summary_df), ("per_class.csv", class_df),
                            ("size_bins.csv", size_df), ("robustness.csv", robust_df)):
        frame.to_csv(RESULTS_DIR / filename, index=False)
    if CFG.report_ground_truth_objects:
        object_count_df.to_csv(RESULTS_DIR / "object_counts_per_image.csv", index=False)
    if CFG.save_comparison_images:
        comparisons_df.to_csv(RESULTS_DIR / "comparison_images_manifest.csv", index=False)
    del outputs, preds, timings
    gc.collect()

assert len(summary_df) == 25
if CFG.save_comparison_images:
    assert len(comparisons_df) == 25 * len(eval_selection)
for metric in ("all_AP", "all_AP50", "Recall@0.5", "mean_pipeline_ms"):
    grid = summary_df.pivot(index="train_overlap", columns="eval_overlap", values=metric)
    grid.to_csv(RESULTS_DIR / f"overlap_grid_{metric.replace('@', '_')}.csv")

shown_columns = ["condition", "train_overlap", "eval_overlap", "all_AP", "all_AP50", "base_AP",
                 "novel_AP", "Precision@0.5", "Recall@0.5", "F1@0.5", "mean_pipeline_ms", "mean_tiles"]
if CFG.report_ground_truth_objects:
    shown_columns += ["ground_truth_objects", "predicted_objects", "correctly_detected_objects", "missed_objects"]
display(summary_df[shown_columns])
print("Completed: 5 training runs / 25 evaluation conditions.")
print("Comparison images:", RESULTS_DIR / "comparisons")


## 7 · Save the 25-condition grid and all comparison images locally

The original weights/results parent folders and ClearML project are unchanged. Each training overlap has distinct best/last checkpoints and history. Results contain 25-row summary tables, 5 × 5 metric CSVs, COCO/raw prediction files, latency files, GT counts and a complete comparison-image manifest. Comparison images are under `comparisons/train_overlap_XX__eval_overlap_YY/<original relative image path>.jpg`. ClearML receives compact aggregate tables/scalars and a small inventory referencing the image directory and manifest, not thousands of image uploads. Completed inference conditions are cached with checkpoint/config/protocol checks.


In [ ]:
# RESULTS_DIR was created in the persistent local project folder at the start.
tables = {"summary.csv": summary_df, "per_class.csv": class_df,
          "size_bins.csv": size_df, "robustness.csv": robust_df}
if CFG.report_ground_truth_objects:
    tables["object_counts_per_image.csv"] = object_count_df
if CFG.save_comparison_images:
    tables["comparison_images_manifest.csv"] = comparisons_df
for filename, frame in tables.items():
    frame.to_csv(RESULTS_DIR / filename, index=False)

def write_json(filename, value):
    destination = RESULTS_DIR / filename
    destination.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
                           encoding="utf-8")
    return destination

write_json("coco_ground_truth.json", coco_gt.dataset)
write_json("config_and_protocol.json", {"config": asdict(CFG), "run_id": RUN_ID,
          "clearml_task_id": task.id, "protocol_dataset_id": PROTOCOL_ID,
          "protocol_fingerprint": FINGERPRINT, "category_ids": CATEGORY_IDS,
          "category_names": CATEGORY_NAMES, "prompts": PROMPTS,
          "base_names": BASE_NAMES, "novel_names": NOVEL_NAMES,
          "eval_image_relpaths": [r["image_relpath"] for r in eval_selection],
          "training_image_relpaths": [r["image_relpath"] for r in train_selection],
          "early_stopping_image_relpaths": [r["image_relpath"] for r in val_selection],
          "platform": platform.platform(), "torch": torch.__version__,
          "ultralytics": ULTRALYTICS_VERSION, "gpu": torch.cuda.get_device_name(0),
          "training_runs": TRAINING_RUNS, "evaluation_conditions": CONDITIONS,
          "comparison_image_threshold": CFG.operating_threshold_yolo})
# Complete per-condition prediction and latency files were already saved incrementally.
assert len(summary_df) == 25
if CFG.save_comparison_images:
    assert len(comparisons_df) == 25 * len(eval_selection)

for path in sorted(RESULTS_DIR.iterdir()):
    if path.is_file():
        LOCAL_INDEX["results"][path.name] = record_local_file(path)

def log_table(title, frame):
    logger.report_table(title=title, series=CFG.eval_split, iteration=0, table_plot=frame)

for name, frame in (("Training overlap vs inference overlap summary", summary_df), ("Per-class AP", class_df),
                    ("Custom size bins", size_df), ("Occlusion / size operating recall", robust_df)):
    log_table(name, frame)
if CFG.report_ground_truth_objects:
    log_table("Ground truth and correctly detected object counts", summary_df[
        ["condition", "train_overlap", "eval_overlap", "ground_truth_objects", "predicted_objects",
         "correctly_detected_objects", "missed_objects", "false_positive_objects", "object_detection_rate"]])
for _, row in summary_df.iterrows():
    for metric in ("all_AP", "all_AP50", "base_AP", "novel_AP", "base_AP50", "novel_AP50",
                   "all_AR500", "Precision@0.5", "Recall@0.5", "F1@0.5", "mean_pipeline_ms", "mean_tiles",
                   *( ("ground_truth_objects", "predicted_objects", "correctly_detected_objects", "missed_objects")
                      if CFG.report_ground_truth_objects else () )):
        if pd.notna(row[metric]):
            logger.report_scalar(title=metric, series=row["condition"], value=float(row[metric]), iteration=0)

# A small local inventory is uploaded to ClearML so saved outputs are discoverable
# without uploading the model checkpoints or prediction dumps themselves.
inventory = {"run_id": RUN_ID, "run_mode": CFG.run_mode,
             "protocol_dataset_id": PROTOCOL_ID, "protocol_fingerprint": FINGERPRINT,
             "clearml_task_id": task.id,
             "local_weights_dir": str(WEIGHTS_DIR),
             "local_results_dir": str(RESULTS_DIR),
             "inventory_path": str(RESULTS_DIR / "local_inventory.json"),
             "objects": LOCAL_INDEX,
             "comparison_images": {"root": str(RESULTS_DIR / "comparisons"),
                                   "count": len(comparisons_df),
                                   "manifest": str(RESULTS_DIR / "comparison_images_manifest.csv")},
             "all_metrics": summary_df.astype(object).where(pd.notna(summary_df), None).to_dict(orient="records")}
manifest_file = RESULTS_DIR / "local_inventory.json"
manifest_file.write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\n", encoding="utf-8")
if manifest_file.stat().st_size >= 2 * 1024 * 1024:
    raise RuntimeError("ClearML inventory unexpectedly exceeds 2 MiB; inspect before uploading.")
task.upload_artifact(name="local_inventory_small", artifact_object=str(manifest_file))
logger.report_text(f"All complete results saved locally: {RESULTS_DIR}")
logger.report_text(f"Protocol Dataset ID: {PROTOCOL_ID} | SHA256 fingerprint: {FINGERPRINT}")
logger.report_text("Metric note: AP uses IoU .50:.95; operating P/R/F1 use conf 0.25, IoU 0.5. "
                   "COCO ignores IDs 0 and 11 without masking detections inside ignored regions.")
logger.flush(wait=True)
task.close()
print("DONE. ClearML project:", CFG.clearml_project)
print("ClearML task:", task.id)
print("Local weights:", WEIGHTS_DIR)
print("Local results:", RESULTS_DIR)
print("Free space (GiB):", round(shutil.disk_usage(ROOT).free / 1024**3, 2))
print("IMPORTANT:", "SMOKE ONLY — do not use for paper" if CFG.run_mode == "smoke" else "Full split evaluated")

print("Training runs / evaluation conditions:", len(TRAINING_RUNS), "/", len(summary_df))
print("Comparison images:", RESULTS_DIR / "comparisons")
